# 01 — First circuit and load flow

**Goal:** run a small source → line → load Case, inspect the solved network, then verify the saved evidence.

**Teaching inputs:** 12.47 kV three-phase system, 1.0 km line, 100 kW load at PF 0.95. These are demonstrator values, not measurements.

**Prediction:** the load-bus voltage should be slightly below the 1.0 pu source voltage.

Run the numbered cells in order. The direct OpenDSS section at the end is optional.

In [1]:
#@title 1. Setup — run once
import contextlib, io, urllib.request, hashlib
_HELPER_URL = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/75b4f394aa096a604123e6e1739d2581e8ef9476/public/notebooks/_lesson.py"
_HELPER_SHA256 = "8618face0c62b85127ffadc7b662bc177ab3ba5a36e32a09ee5223f562e02ba2"
_blob = urllib.request.urlopen(_HELPER_URL, timeout=60).read()
assert hashlib.sha256(_blob).hexdigest() == _HELPER_SHA256, "lesson helper hash mismatch"
_helper_output = io.StringIO()
with contextlib.redirect_stdout(_helper_output):
    exec(compile(_blob, "lesson helper", "exec"))
for _line in _helper_output.getvalue().splitlines():
    if "lesson helpers ready" not in _line.lower():
        print(_line)
print("Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.")

# Compatibility renderer for the currently pinned public wheel; newer wheels use cept.public_notebook.
def display_run_compat(run_dir):
    import html as _html
    from IPython.display import HTML, display
    result = read(Path(run_dir) / 'results.json')
    sld = result.get('sld') or result.get('sld_after') or {}
    nodes, edges = sld.get('nodes') or [], sld.get('edges') or []
    if not nodes:
        display(HTML('<p><strong>CEPT result:</strong> this run does not carry an SLD.</p>'))
        return
    xs, ys = [float(n['x']) for n in nodes], [float(n['y']) for n in nodes]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    dx, dy = max(x1-x0, 1.0), max(y1-y0, 1.0)
    def xy(node):
        return 65 + (float(node['x'])-x0)/dx*870, 45 + (float(node['y'])-y0)/dy*420
    pos = {str(n['id']): xy(n) for n in nodes}
    line_svg = []
    for edge in edges:
        if str(edge.get('src')) in pos and str(edge.get('dst')) in pos:
            a, b = pos[str(edge['src'])], pos[str(edge['dst'])]
            dash = ' stroke-dasharray=\"8 6\"' if edge.get('status') == 'open' else ''
            edge_label = _html.escape(str(edge.get('id', 'branch')))
            line_svg.append(f'<line x1=\"{a[0]:.1f}\" y1=\"{a[1]:.1f}\" x2=\"{b[0]:.1f}\" y2=\"{b[1]:.1f}\" stroke=\"#7a879a\" stroke-width=\"3\"{dash}><title>{edge_label}</title></line>')
    vmin, vmax = float(sld.get('v_min_pu', .95)), float(sld.get('v_max_pu', 1.05))
    bus_svg, rows = [], []
    for node in nodes:
        volts = {int(k): float(v) for k, v in (node.get('v_pu') or {}).items()}
        angles = {int(k): float(v) for k, v in (node.get('angle_deg') or {}).items()}
        values = list(volts.values())
        low, high = any(v < vmin for v in values), any(v > vmax for v in values)
        status = 'NO DATA' if not values else 'OUT' if low and high else 'UNDER' if low else 'OVER' if high else 'OK'
        fill = {'OK':'#e8f5ec','UNDER':'#fff3d9','OVER':'#ffe7e1','OUT':'#f7e7ff','NO DATA':'#eef1f5'}[status]
        stroke = {'OK':'#2f7d4a','UNDER':'#a46700','OVER':'#b8432e','OUT':'#8147a6','NO DATA':'#7b8796'}[status]
        x, y = pos[str(node['id'])]
        phase = lambda p: '—' if p not in volts else f'{volts[p]:.4f} pu' + (f' @ {angles[p]:.2f}°' if p in angles else '')
        tip = _html.escape('Bus '+str(node['id'])+'\nStatus: '+status+'\n'+'\n'.join(f'{label}: {phase(p)}' for p,label in [(1,'A'),(2,'B'),(3,'C')] if p in volts))
        label = _html.escape(str(node['id']))
        bus_svg.append(f'<g tabindex=\"0\"><title>{tip}</title><rect x=\"{x-31:.1f}\" y=\"{y-11:.1f}\" width=\"62\" height=\"22\" rx=\"5\" fill=\"{fill}\" stroke=\"{stroke}\" stroke-width=\"2\"/><text x=\"{x:.1f}\" y=\"{y+4:.1f}\" text-anchor=\"middle\" font-size=\"12\" font-weight=\"700\">{label}</text></g>')
        rows.append('<tr><th>'+label+'</th><td>'+phase(1)+'</td><td>'+phase(2)+'</td><td>'+phase(3)+'</td><td><strong>'+status+'</strong></td></tr>')
    display(HTML('<div style=\"font-family:system-ui,sans-serif\"><h3>Interactive CEPT SLD</h3><p style=\"color:#657187\">Hover or focus a bus for solver-returned values.</p><div style=\"overflow:hidden;border:1px solid #d9dee8;border-radius:10px\"><svg viewBox=\"0 0 1000 510\" style=\"width:100%;height:auto;display:block\">'+''.join(line_svg)+''.join(bus_svg)+'</svg></div><div style=\"overflow-x:auto;margin-top:10px\"><table style=\"border-collapse:collapse;width:100%;min-width:650px\"><thead><tr><th>Bus</th><th>Phase A</th><th>Phase B</th><th>Phase C</th><th>Status</th></tr></thead><tbody>'+''.join(rows)+'</tbody></table></div></div>'))


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version
cept-power-studio 0.2.0.dev0
Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


In [2]:
#@title 2. Inputs — source, line, and load
CASE_PATH = WORKSPACE / "first_circuit_case.json"
CASE_PAYLOAD = first_circuit_case()
CASE_PATH.write_text(json.dumps(CASE_PAYLOAD, indent=2) + "\n", encoding="utf-8")
RUN_DIR = WORKSPACE / "runs" / "01-first-circuit"

table(
    ["declared input", "value", "unit"],
    [
        ("frequency", 60, "Hz"),
        ("line length", 1.0, "km"),
        ("load", 100.0, "kW"),
        ("load power factor", 0.95, "1"),
    ],
)

| declared input | value | unit |
| --- | --- | --- |
| frequency | 60 | Hz |
| line length | 1.0 | km |
| load | 100.0 | kW |
| load power factor | 0.95 | 1 |


In [3]:
# 3. Run study — same CEPT CLI as a normal terminal
!cept study run first_circuit_case.json \
    --out runs/01-first-circuit \
    --force \
    --format text

CEPT study result: FINISHED
----------------------------
Result             Finished the study and saved the evidence
Saved run          runs\01-first-circuit
Case fingerprint   aaea341ddd26 (matches the case you ran)

What this means
  The study completed and saved solver-backed evidence.
  It does NOT approve a real project or field installation.

Next
  cept study verify runs\01-first-circuit --format text


In [4]:
#@title 4. Explore — SLD and bus status
RUN_DIR = WORKSPACE / "runs" / "01-first-circuit"
try:
    from cept.public_notebook import display_run
except ModuleNotFoundError:
    display_run = display_run_compat
display_run(RUN_DIR)

Bus,Phase A,Phase B,Phase C,Status
grid,1.0000 pu @ 0.00°,1.0000 pu @ -120.00°,1.0000 pu @ 120.00°,OK
load,0.9998 pu @ -0.01°,0.9998 pu @ -120.01°,0.9998 pu @ 119.99°,OK
source,1.0000 pu @ 0.00°,1.0000 pu @ -120.00°,1.0000 pu @ 120.00°,OK


In [5]:
#@title 5. Engineering result — persisted solver values
results = read(RUN_DIR / "results.json")
load_flow = results["load_flow"]
table(
    ["quantity", "value", "unit"],
    [
        ("converged", load_flow["converged"], "bool"),
        ("total load", load_flow["total_load_kw"], "kW"),
        ("total loss", load_flow["total_loss_kw"], "kW"),
        ("source P", load_flow["source_p_kw"], "kW"),
    ],
)
assert load_flow["converged"] is True

| quantity | value | unit |
| --- | --- | --- |
| converged | True | bool |
| total load | 100.0 | kW |
| total loss | 0.0143 | kW |
| source P | 100.0141 | kW |


In [6]:
# 6. Verify — check this exact saved run
!cept study verify runs/01-first-circuit --format text

CEPT study check: PASSED
----------------------------
Study              Load flow (OpenDSS)
Case fingerprint   aaea341ddd26 (matches the case you ran)

Checked   3 groups, 14 checks, all passed
  [PASS] Case identity (4 checks)
  [PASS] Solver result (3 checks)
  [PASS] Saved evidence (7 checks)

What this means
  The saved result matches its Case, solver run, and saved evidence.
  It does NOT approve a real project or field installation.

Saved evidence     runs\01-first-circuit\public-verification.json

For the full check list
  cept study verify runs\01-first-circuit --format json


## 7. Interpret

The load flow converged and the SLD/bus table show the solver-backed operating point for this declared demonstrator.

**What this proves:** this CEPT workflow ran, saved its result, and can be verified against its Case and artifacts.

**What this does not prove:** acceptance of a real feeder or field installation.

**Try next:** change one declared input in CASE_PAYLOAD, predict the direction of change, restart the kernel, and rerun the lesson.

## Optional — direct OpenDSS comparison

The cells below are for learners who want to inspect the solver route. They are not required for the main CEPT workflow. Run them in order to compare direct OpenDSS readback with the persisted CEPT result.

In [7]:
#@title Under the hood - direct OpenDSS solve (optional)
import opendssdirect as dss

for command in [
    'Clear',
    'New Circuit.first basekv=12.47 pu=1.0 phases=3 bus1=source',
    'New Line.line1 bus1=source.1.2.3 bus2=load.1.2.3 phases=3 length=1 units=km r1=0.2 x1=0.4 r0=0.6 x0=1.2 c1=0 c0=0',
    'New Load.load1 bus1=load.1.2.3 phases=3 conn=wye kv=12.47 kw=100 pf=0.95',
    'CalcVoltageBases',
    'Solve',
]:
    dss.Text.Command(command)
assert dss.Solution.Converged()
print("Direct OpenDSS solve finished: the circuit converged.")


Direct OpenDSS solve finished: the circuit converged.


In [8]:
#@title Under the hood — direct OpenDSS readback (optional)
dss.Circuit.SetActiveBus('load')
direct_values = dss.Bus.puVmagAngle()
direct_by_phase = {phase: float(direct_values[2 * (phase - 1)]) for phase in (1, 2, 3)}
table(['source', 'phase', 'voltage magnitude', 'unit'], [('direct OpenDSS', phase, direct_by_phase[phase], 'pu') for phase in (1, 2, 3)])
assert all(value > 0 for value in direct_by_phase.values())

| source | phase | voltage magnitude | unit |
| --- | --- | --- | --- |
| direct OpenDSS | 1 | 0.9997586726902581 | pu |
| direct OpenDSS | 2 | 0.9997586726902756 | pu |
| direct OpenDSS | 3 | 0.9997586726902107 | pu |


In [9]:
#@title Compare solver outputs (optional details)
results = read(RUN_DIR / "results.json")
verify_summary = read(RUN_DIR / "public-verification.json")
cept_rows = [row for row in results["load_flow"]["bus_voltages"] if row["bus"].lower() == "load"]
cept_by_phase = {row["phase"]: row["v_pu"] for row in cept_rows}
table(
    ["phase", "direct OpenDSS pu", "CEPT pu", "|difference| pu"],
    [(phase, direct_by_phase[phase], cept_by_phase[phase], abs(direct_by_phase[phase] - cept_by_phase[phase])) for phase in (1, 2, 3)],
)
max_abs_diff_pu = max(abs(direct_by_phase[phase] - cept_by_phase[phase]) for phase in (1, 2, 3))
print()
print("Direct OpenDSS and CEPT agree")
print("-----------------------------")
print(f"Result        {'PASSED' if verify_summary['passed'] else 'Needs attention'}")
print(f"Largest phase difference: {max_abs_diff_pu:.2e} per unit (limit 1e-4 per unit)")
print("A small difference is rounding, not a different answer.")
assert verify_summary["status"] == "PASS"
assert verify_summary["passed"] is True
assert set(cept_by_phase) == {1, 2, 3}
assert max_abs_diff_pu < 1e-4


| phase | direct OpenDSS pu | CEPT pu | |difference| pu |
| --- | --- | --- | --- |
| 1 | 0.9997586726902581 | 0.999787 | 2.8327309741893458e-05 |
| 2 | 0.9997586726902756 | 0.999787 | 2.8327309724351935e-05 |
| 3 | 0.9997586726902107 | 0.999787 | 2.832730978929998e-05 |

Direct OpenDSS and CEPT agree
-----------------------------
Result        PASSED
Largest phase difference: 2.83e-05 per unit (limit 1e-4 per unit)
A small difference is rounding, not a different answer.
